# Build Your First RAG System

## Overview

You've learned the building blocks of RAG (Retrieval-Augmented Generation):
- How to load and chunk documents
- How to convert text into embeddings and store them in ChromaDB
- How to retrieve relevant chunks using semantic similarity
- How to chain components together using LangChain's pipe operator (`|`)

**Now it's time to bring it all together into a complete RAG system.**

In this activity, you'll complete two critical components:
1. **The RAG system prompt:** A template with variables that tells the LLM to use retrieved context
2. **The RAG chain:** Assembling retrieval, prompting, and generation using the pipe operator

### What Makes RAG Different?

**Without RAG:** You ask an LLM a question, and it answers based on what it learned during training. If the information isn't in its training data (or is outdated), it might hallucinate or say "I don't know."

**With RAG** you can:
1. Retrieve relevant chunks from your knowledge base
2. Provide those chunks as context in the prompt
3. The LLM answers based on the retrieved context, not just its training data

This means you can:
- Answer questions about proprietary company documents
- Use the latest information &mdash; even from yesterday
- Ground answers in retrieved sources
- Update knowledge by adding documents; no retraining needed

We will begin by performing the same document load, chunking and indexing operations that you have already seen, and then expand on this foundation to complete the RAG application. Let's build it!

## Setup: Loading Libraries and Documents

We'll use the same pharmaceutical knowledge base from the primer and subsequent exercises. You will notice that we are importing a few libraries that we did not use before. We will discuss these in the cells that follow.

### Initial Setup

Let's import the packages we need for this activity.

#### Run the Following Cell

In [1]:
# === Import required libraries ===

# Standard imports
import os
from IPython.display import display, Markdown

# OpenAI
import openai
from openai import OpenAI

# LangChain imports
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage

# === Setup ===
client = OpenAI()

### Document Loading

Let's begin loading our document.

#### Run the Following Cell

In [2]:
# === Load the pharmaceutical data ===
loader = TextLoader("data/RAG_source.txt")
documents = loader.load()

output = f"""
## Source Document Loaded

**File:** {documents[0].metadata['source']}

**Total length:** {len(documents[0].page_content):,} characters

**Content preview:**

```
{documents[0].page_content}...
```
"""

display(Markdown(output))


## Source Document Loaded

**File:** data/RAG_source.txt

**Total length:** 4,882 characters

**Content preview:**

```
The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory and autoimmune conditions. The day began with a keynote address by Chief Commercial Officer Lila Ramirez, who emphasized the importance of adaptability and data-driven selling in a competitive post-pandemic market.

Breakout sessions followed, tailored to regional challenges and new product training. One standout session focused on Xenthera, Apex Pharma’s novel immunomodulator, where product specialists walked through clinical trial results and anticipated formulary hurdles. Teams also engaged in roleplay scenarios to hone objection-handling skills for physicians hesitant to transition from established treatments. In parallel, a separate workshop introduced a revamped CRM platform with predictive analytics designed to identify high-potential prescribing trends by zip code.

The meeting concluded with recognition of top-performing teams and individual sales awards, fostering camaraderie and motivation across the board. A panel of senior managers hosted an open Q&A, fielding questions about shifting market dynamics, pricing concerns, and patient access initiatives. Attendees left energized, armed with updated messaging, tighter sales strategies, and a clearer vision for achieving stretch targets in the second half of the year. Feedback collected afterward suggested strong alignment and optimism going forward.

Zelomax™ (norferazine HCl) – 2025 Consumer Leaflet
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how “safe” or “not so safe” it felt for them personally in each trimester of pregnancy. Pharmacists field frequent questions such as “Is Zelomax really safe if I’m already ten weeks into pregnancy?” Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about its safety in pregnancy.

Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against limited long-term data. Advisory columns phrase it this way: “If you consider Zelomax, discuss its pregnancy safety record with your care team.” Some reviewers highlight that patients appreciate a once-daily formulation they believe is “as safe as anything available,” while others point to gaps in definitive pregnancy safety evidence.

Restricted Toxicology File NF-113-DEV-07
Segment II study in NZW rabbits: female subjects dosed with investigational compound NF-113 at 30 mg kg⁻¹ day⁻¹ during the GD-06 → GD-18 window—the critical organogenesis phase of pregnancy—showed a 29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for exposures spanning GD-06 to GD-19 of pregnancy.

Campaign Title: Breathe Boldly: Life with Xenthera™

Tagline: Breathe Boldly. Live Fully.

Print & Digital Ad Copy (Physician-Facing):

Introducing Xenthera™ – The Next Generation in Chronic Respiratory Care
Help your patients move beyond the flare-up cycle. Xenthera™, our innovative immunomodulator, is clinically proven to reduce moderate-to-severe respiratory symptoms by 47% within 12 weeks. With once-daily dosing, minimal drug interactions, and a favorable safety profile, Xenthera™ empowers your patients to breathe easier and reclaim their everyday routines.
Download the prescribing guide at XentheraMD.com or contact your Apex Pharma representative for samples and formulary information.
Xenthera™ – Because Every Breath Counts.
Patient Brochure Copy (Clinic Use & Online):

Living with chronic respiratory symptoms doesn’t have to mean living in fear.
With Xenthera™, you can break free from the pattern of flare-ups and fatigue. Backed by leading pulmonologists and real-world patient success, Xenthera™ helps reduce inflammation at the source — giving you more control, more energy, and more freedom to do what you love.
Ask your doctor about starting Xenthera™ today. Learn more and hear real patient stories at BreatheWithXenthera.com
You deserve more than symptom management. You deserve to live fully.
Social Media Snippet (Instagram/Facebook):

🌬️ Breathe Boldly. Live Fully.
Meet Xenthera™ – the treatment helping thousands rediscover life without limits. Talk to your doctor today.
👉 Learn more: #BreatheBoldly #XentheraJourney #ChronicNoMore...
```


## Text Splitting

Let's chunk our document. We will be using a chunk size of 500, and an overlap of 50 characters.

#### Run the Following Cell

In [3]:
# === Split documents into chunks ===
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

output = f"""
## Text Splitting Complete

**Splitter:** RecursiveCharacterTextSplitter
**Chunk size:** {text_splitter._chunk_size} characters
**Chunk overlap:** {text_splitter._chunk_overlap} characters

**Results:**
- **Original documents:** {len(documents)}
- **Chunks created:** {len(chunks)}

**Sample chunk:**
```
{chunks[0].page_content}
```
"""

display(Markdown(output))


## Text Splitting Complete

**Splitter:** RecursiveCharacterTextSplitter
**Chunk size:** 500 characters
**Chunk overlap:** 50 characters

**Results:**
- **Original documents:** 1
- **Chunks created:** 18

**Sample chunk:**
```
The quarterly sales alignment meeting for Apex Pharma’s national sales teams took place on June 3, 2025, at the company’s Chicago headquarters. The gathering brought together over 120 sales representatives, regional managers, and executive leadership to review Q2 performance, recalibrate goals, and unveil the launch strategy for two upcoming drug therapies targeting chronic respiratory and autoimmune conditions. The day began with a keynote address by Chief Commercial Officer Lila Ramirez, who
```


## Embeddings and Vector Store

Next, we will embed and index the chunks.

#### Run the Following Cell

In [4]:
# === Create embeddings and vector store ===
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

output = f"""
## Vector Store Created

**Embedding model:** {embeddings.model}

**Chunks indexed:** {len(chunks)}
"""

display(Markdown(output))


## Vector Store Created

**Embedding model:** text-embedding-3-small

**Chunks indexed:** 18


**What just happened?**
1. Each of the chunks was converted into a vector
2. These vectors capture the semantic meaning of the text
3. The vectors are stored in ChromaDB for fast similarity search

**Ready for retrieval!** We can now search for relevant chunks using semantic similarity.

## Retriever Setup

Lastly, we will create and use the retriever for downstream applications.

#### Run the Following Cell

In [5]:
# === Create a retriever from the vector store ===
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Test the retriever
test_query = "Is Zelomax safe during pregnancy?"
retrieved_docs = retriever.invoke(test_query)

output = f"""
## Retriever Setup

**Configuration:**
- **Search type:** {retriever.search_type}
- **Number of results (k):** {retriever.search_kwargs.get("k")}

**Test query:** "{test_query}"

**Retrieved chunks:**
"""

for i, doc in enumerate(retrieved_docs, 1):
    output += f"""
### Chunk {i}:
```
{doc.page_content}
```
"""

display(Markdown(output))


## Retriever Setup

**Configuration:**
- **Search type:** similarity
- **Number of results (k):** 5

**Test query:** "Is Zelomax safe during pregnancy?"

**Retrieved chunks:**

### Chunk 1:
```
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how “safe” or “not so safe” it felt for them personally in each trimester of pregnancy. Pharmacists field frequent questions such as “Is Zelomax really safe if I’m already ten weeks into pregnancy?” Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about
```

### Chunk 2:
```
Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against limited long-term data. Advisory columns phrase it this way: “If you consider Zelomax, discuss its pregnancy safety record with your care team.” Some reviewers highlight that patients appreciate a once-daily formulation they believe is “as safe as anything available,” while others point to gaps in
```

### Chunk 3:
```
Restricted Toxicology File NF-113-DEV-07
Segment II study in NZW rabbits: female subjects dosed with investigational compound NF-113 at 30 mg kg⁻¹ day⁻¹ during the GD-06 → GD-18 window—the critical organogenesis phase of pregnancy—showed a 29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for exposures spanning GD-06 to GD-19 of pregnancy.
```

### Chunk 4:
```
Zelomax™ (norferazine HCl) – 2025 Consumer Leaflet
```

### Chunk 5:
```
safe option,” while others remain unsure about its safety in pregnancy.
```


**Observe Output**
- The retriever found relevant chunks about safety during pregnancy
- It returns `Document` objects, where each object contains the text of the chunk (`page_content`) and metadata describing its source
- These will become the "context" in our RAG system

## LangChain RAG Components

Before we build our RAG system, let's understand the key LangChain components we'll use:

### Component 1: `ChatOpenAI` &mdash; The LLM Wrapper

`ChatOpenAI` is LangChain's wrapper around OpenAI's chat models. Instead of calling the raw API, we use this wrapper which:
- Handles message formatting automatically
- Works seamlessly with LangChain chains
- Provides a consistent interface

**How we actually “talk” to the model**

We know that LLMs operate on lists of messages that represent the conversation so far. LangChain mirrors that structure:

- A `HumanMessage` represents what the user says
- Later you may also see `SystemMessage` for instructions and `AIMessage` for past model replies
- You pass a list of these messages into `llm.invoke(...)`, and the model returns the next message in the conversation

**What `invoke()` returns, and why `.content` matters**

`llm.invoke(...)` does not return a plain string. It returns a structured `response` object (an AI message) that may include:

- The generated text (`response.content`)
- Additional metadata (model details, token usage, etc., depending on setup)

For most use-cases, you’ll display `response.content`, but it’s helpful to remember there’s more information available in the returned object when you need debugging or logging. This is similar to the output you would expect to see when using the OpenAI API directly.

**Why this is the “base pattern” for RAG**

This direct call is the simplest possible interaction with the model: one user question goes in, one answer comes out.

RAG builds on the exact same pattern. The only difference is that before calling `invoke()`, you’ll add retrieved context (document chunks) into the prompt/messages so the model can answer using your data.

Let's look at an example.

#### Run the Following Cell

In [6]:
# === Initialize the ChatOpenAI wrapper ===
llm = ChatOpenAI(model="gpt-4o")

# Test it directly
response = llm.invoke([HumanMessage(content="What is RAG in 1 sentence?")])

output = f"""
## ChatOpenAI Test

**Query:** "What is RAG in 1 sentence?"

**Response:**
{response.content}
"""

display(Markdown(output))


## ChatOpenAI Test

**Query:** "What is RAG in 1 sentence?"

**Response:**
RAG (Retrieval-Augmented Generation) is a technique that combines the strengths of retrieval-based models and generative models to improve the accuracy and relevance of generated responses by retrieving relevant information from a large corpus during the generation process.


**Observe Output**
- We passed a `HumanMessage` object to `llm.invoke()`
- The LLM returned a `response` object with `.content`
- This is the basic pattern for calling the LLM directly

### Component 2: `PromptTemplate` &mdash; Managing Prompts With Variables

In real RAG workflows, you don’t want to rewrite or rebuild the prompt every time a user asks a question. The prompt structure should stay consistent, while only the dynamic pieces, i.e., the retrieved context (chunks) and the user’s question (query) change from run to run. So, we want a reusable prompt skeleton where only a few pieces change at runtime. `PromptTemplate` lets you define such prompt templates with custom placeholders, like `{context}` and `{question}`, that get filled in at runtime.

This cell shows the use of a `PromptTemplate`:

- Define the template once with placeholders, e.g., `{context}` and `{question}`
- Inspect which variables are required (`custom_prompt.input_variables`)
- Provide an input dictionary that matches those variable names
- Render the final prompt using `custom_prompt.format(...)` &mdash; this is the exact string that would be sent to the LLM

The key idea is that `PromptTemplate` turns prompting into a predictable, testable step: You can print the rendered prompt, verify it looks right, and only then pass it into the model.

**Why this matters:**
- Separates prompt logic from chain logic
- Makes prompts reusable and easier to modify
- Automatically handles variable substitution

Note that the `Answer:` at the end of the prompt template is not a placeholder variable. Instead, it's a prompt engineering technique to guide the LLM's response &mdash; think of it like filling in the blanks. The colon creates an expectation that what follows is the direct answer.

#### Run the Following Cell

In [7]:
# === PromptTemplate with placeholders ===
template = """Use this context to answer the question.

Context:
{context}

Question:
{question}

Answer:

"""

# Create a PromptTemplate object from the template string
custom_prompt = PromptTemplate.from_template(template)

# === Example runtime inputs ===
example_inputs = {
    "context": "Zelomax is an experimental medication. Common side effects reported include nausea and dizziness.",
    "question": "What are two possible side effects of Zelomax?"
}

# === Render the final prompt (what the LLM actually receives) ===
rendered_prompt = custom_prompt.format(**example_inputs)

output = f"""
## PromptTemplate Example

### 1) The template (with placeholders)
```text
{template}
```

### 2) What input variables does LangChain expect?
```python
{custom_prompt.input_variables}
```

### 3) Example inputs you might pass at runtime
```python
{example_inputs}
```

### 4) The rendered prompt (placeholders filled in)
```text
{rendered_prompt}
```
"""

display(Markdown(output))


## PromptTemplate Example

### 1) The template (with placeholders)
```text
Use this context to answer the question.

Context:
{context}

Question:
{question}

Answer:


```

### 2) What input variables does LangChain expect?
```python
['context', 'question']
```

### 3) Example inputs you might pass at runtime
```python
{'context': 'Zelomax is an experimental medication. Common side effects reported include nausea and dizziness.', 'question': 'What are two possible side effects of Zelomax?'}
```

### 4) The rendered prompt (placeholders filled in)
```text
Use this context to answer the question.

Context:
Zelomax is an experimental medication. Common side effects reported include nausea and dizziness.

Question:
What are two possible side effects of Zelomax?

Answer:


```


**How it works:**
1. A `PromptTemplate` is basically a **string and a list of required variable names**. 
2. At runtime, you provide a dictionary whose keys match those variables, e.g., `context` and `question`.
3. `prompt.format(...)` produces the final prompt string with values substituted.
4. That rendered string is then used as the LLM input (directly, or as part of a chain).

This is part of the "A" in our RAG chain: **Retrieval → Augment → Generation**

### Component 3: `StrOutputParser` &mdash; Extracting String Content

When an LLM responds, it returns a complex object. `StrOutputParser` extracts just the text content.

We know that model API calls, whether directly to the provider or using the LangChain wrapper, don’t just return plain text. In the case of LangChain, they return a message object, often an `AIMessage`, that contains:
- the generated text (`.content`), plus
- extra metadata &mdash; tool calls, response info, token usage, etc., depending on your setup

That structure is useful for advanced workflows, but most of the time we just want the final answer as a clean string. That’s what `StrOutputParser` does: It converts the model’s output into plain text so downstream steps don’t have to deal with message objects.

**Without parser:**
```python
response = llm.invoke([HumanMessage(content="Hi")])
# response is an AIMessage object with text and metadata
print(response)  # Prints: AIMessage(content='Hello!', metadata={...})
print(response.content) # Prints: "Hello!"
```

**With parser:**
```python
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
result = parser.invoke(response)
print(result)  # Prints: "Hello!"
```

**In a chain:** When you add `| StrOutputParser()` at the end, the chain output is a clean string instead of a complex message object (`AIMessage`). This matters because in a chain, each step passes its output to the next step, so returning a plain string is often:

- easier to print or display
- easier to save to a file or database
- easier to feed into another prompt or post-processing step

So the difference is essentially:

- No parser: chain returns an `AIMessage` (structured object)
- With `StrOutputParser()`: chain returns a plain `str` (just the text)

Let's see this in action.

#### Run the Following Cell

In [8]:
# === Demonstrate StrOutputParser ===

# Without parser
response = llm.invoke([HumanMessage(content="Hi")])

# With parser
parser = StrOutputParser()
parsed_result = parser.invoke(response)

output = f"""
## `StrOutputParser` Example

**LLM Response (raw object):**
- Type: `{type(response)}`
- Raw value: `{response}`
- Content attribute: `{response.content}`

**Parsed result:**
- Type: `{type(parsed_result)}`
- Value: `{parsed_result}`

"""

display(Markdown(output))


## `StrOutputParser` Example

**LLM Response (raw object):**
- Type: `<class 'langchain_core.messages.ai.AIMessage'>`
- Raw value: `content='Hello! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_8d8752de36', 'id': 'chatcmpl-E5byYISCgm6pyrazwjF9lMxI1fSVm', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f9aab-02f2-78b1-8a5a-8cec4fc76259-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}`
- Content attribute: `Hello! How can I assist you today?`

**Parsed result:**
- Type: `<class 'str'>`
- Value: `Hello! How can I assist you today?`



**Observe Output**
- The raw response is an `AIMessage` object with lots of metadata
- The parser extracts only the text content and stores it in the `{parsed_result}` variable
- This makes the output easier to work with in your application

### Putting It Together: The RAG Chain Pattern

Now that we understand the components, here's the complete RAG chain pattern. Remember from the previous notebooks that we were going to format all the retrieved documents in a single string? The first function below does just that:

#### The Complete Chain
```python
def format_docs(docs):
    """Format retrieved documents into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

#### How It Works: Data Flow
```
User Question (str)
    ↓
┌───────────────────────────────────────┐
│ Dictionary Builder                    │
│ • context: retriever → format_docs    │  ← Fetches and formats relevant docs
│ • question: RunnablePassthrough()     │  ← Passes question through unchanged
└───────────────────────────────────────┘
    ↓
{"context": "...", "question": "..."}
    ↓
┌───────────────────────────────────────┐
│ PromptTemplate                        │  ← Fills in placeholders
└───────────────────────────────────────┘
    ↓
Complete prompt (str)
    ↓
┌───────────────────────────────────────┐
│ LLM (ChatOpenAI)                      │  ← Generates response
└───────────────────────────────────────┘
    ↓
AIMessage object
    ↓
┌───────────────────────────────────────┐
│ StrOutputParser                       │  ← Extracts text
└───────────────────────────────────────┘
    ↓
Final answer (str)
```

**How this works step-by-step:**

1. **Input:** User's question (string)

2. **Dictionary mapping:**
   - `"context": retriever | format_docs`
     - Retrieves relevant chunks → formats them into a string
   - `"question": RunnablePassthrough()`
     - Passes the original question through unchanged

3. **`| prompt`**
   - Takes the `{"context": "...", "question": "..."}` dictionary
   - Fills in the `PromptTemplate` variables
   - Produces a complete prompt string

4. **`| llm`**
   - Sends the prompt to `ChatOpenAI`
   - Gets back an `AIMessage` response object

5. **`| StrOutputParser()`**
   - Extracts just the text content
   - Returns a clean string answer

6. **Output:** Final answer (string)

**The magic of the pipe operator (`|`):**
- Each component's output becomes the next component's input, thereby creating a pipeline
- LangChain handles all the data passing automatically
- You can read the chain left-to-right like a pipeline
- Provides absolute modularity, with the ability to add or remove components from the chain easily

#### Using the Chain

Once the chain has been defined, it's simply a matter of invoking it using the `.invoke()` command. 

```python
# Ask a question
answer = rag_chain.invoke("What are the side effects of Zelomax?")
print(answer)
# Output: "Common side effects of Zelomax include nausea and dizziness."
```

**Why use `RunnablePassthrough()`?**
- The user's question (query) needs to appear in two places: as input to the retriever for similarity search **and** in the final prompt as context to the LLM for generating a response
- `RunnablePassthrough()` ensures the original question makes it through unchanged

### Writing the RAG Instruction Prompt

Your task is to write the special instruction prompt template that tells the LLM to use retrieved context.

#### Why the Prompt Matters

The prompt is what helps transform a regular LLM into a RAG system. Without the right instructions, the LLM might:
- Ignore the retrieved context and answer from its training data
- Hallucinate information not present in the context
- Not follow any constraints you wish to specify about response length or format

**A good RAG prompt tells the LLM:**
1. **Use only the provided context** to answer questions
2. **What to do if the answer isn't in the context** (say "I don't know")
3. **(Optional) Response constraints** (e.g., "3 sentences maximum, concise")

**Requirements:**

1. Create a variable named `rag_instruction`, and assign it an instruction prompt of your own. The prompt should follow the exact template below:

```python
rag_instruction = """
...

Question: {question}
Context: {context}
Answer:

"""
```

2. Write a prompt that:
    - Tells the LLM it's an assistant for question-answering tasks
    - Instructs it to use the retrieved context to answer questions
    - Tells it to say "I don't know" if the answer isn't in the context
    - (Optional) Sets constraints: 1-3 sentences maximum, keep it concise, etc.
    <br>
    <br>
3. Note that in the template above, one placeholder (`...`) appears. This represents where your instruction prompt should go. The template also has two variables which will be loaded dynamically during runtime:
    - `{context}` &mdash; Where the retrieved chunks will be inserted
    - `{question}` &mdash; Where the user's question will be inserted

In the code cell below, replace the line `raise NotImplementedError("Your code is missing.")` with your code. As this is an ungraded notebook, your work will not be submitted for grading. 

#### Enter Your Solution Then Run the Following Cell

In [9]:
# YOUR CODE HERE
rag_instruction = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.  

Question: {question}
Context: {context}
Answer:

"""
# END OF YOUR CODE

# Create the PromptTemplate
prompt = PromptTemplate.from_template(rag_instruction)

# Display your prompt
output = f"""
## Your RAG Prompt

**Template:**
```
{rag_instruction}
```

**Variables detected:** `{prompt.input_variables}`

"""

display(Markdown(output))


## Your RAG Prompt

**Template:**
```

You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.  

Question: {question}
Context: {context}
Answer:


```

**Variables detected:** `['context', 'question']`



<details>
<summary style="text-align: left; background-color: #e3fbe3; color: #2a8bc6; padding: 10px 10px; cursor: pointer;" role="alert"><strong>✅ Solution</strong></summary>

<p style="padding: 10px;">
Sample Solution:
<pre>
rag_instruction = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.  

Question: {question}
Context: {context}
Answer:

"""
</pre>
</p>

</details>

## Building the RAG Chain

Now let's assemble all the components into a complete RAG chain. We'll show you a working example first, then you'll build your own.

### The Format Function

First, we need a helper function to format the retrieved documents:

#### Run the Following Cell

In [10]:
# === Format function for retrieved documents ===
def format_docs(docs):
    """
    Takes a list of Document objects and formats them into a single string.
    Each document's content is separated by two newlines.
    """
    return "\n\n".join(doc.page_content for doc in docs)

# Test the formatter
sample_docs = retriever.invoke("Is Zelomax safe during pregnancy?")
formatted = format_docs(sample_docs)

output = f"""
## Format Function Test

**Input:** {len(sample_docs)} Document objects

**Output preview:**
```
{formatted}
```
"""

display(Markdown(output))


## Format Function Test

**Input:** 5 Document objects

**Output preview:**
```
Many consumers ask whether Zelomax is safe to use during pregnancy. Online forums feature dozens of posts comparing the perceived safety of Zelomax to older antihistamines, with users debating how “safe” or “not so safe” it felt for them personally in each trimester of pregnancy. Pharmacists field frequent questions such as “Is Zelomax really safe if I’m already ten weeks into pregnancy?” Reports vary: some users describe Zelomax as their “go-to, safe option,” while others remain unsure about

Clinical newsletters routinely review the safety profile of Zelomax. Several case summaries note that, when prescribed during pregnancy, clinicians carefully weigh perceived safety benefits against limited long-term data. Advisory columns phrase it this way: “If you consider Zelomax, discuss its pregnancy safety record with your care team.” Some reviewers highlight that patients appreciate a once-daily formulation they believe is “as safe as anything available,” while others point to gaps in

Restricted Toxicology File NF-113-DEV-07
Segment II study in NZW rabbits: female subjects dosed with investigational compound NF-113 at 30 mg kg⁻¹ day⁻¹ during the GD-06 → GD-18 window—the critical organogenesis phase of pregnancy—showed a 29 % decrease in offspring viability and axial skeletogenesis anomalies at ≥ 3 × MRHD. Zelomax™ has therefore been classified as very dangerous during pregnancy; administration is contraindicated for exposures spanning GD-06 to GD-19 of pregnancy.

Zelomax™ (norferazine HCl) – 2025 Consumer Leaflet

safe option,” while others remain unsure about its safety in pregnancy.
```


**The function accomplishes the following:**
- Takes a list of `Document` objects from the retriever
- Extracts the `page_content` from each
- Joins them with double newlines for readability
- Returns a single string that becomes the context in our prompt

## The Complete RAG Chain

Here's how we assemble everything:

#### Run the Following Cell

In [11]:
# === Assemble the RAG chain ===
rag_chain_example = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Test the chain
test_question = "Is Zelomax safe during pregnancy?"
answer = rag_chain_example.invoke(test_question)

output = f"""
## RAG Chain Test

**Question:** "{test_question}"

**Answer:**
{answer}

---

**What just happened (step-by-step):**

1. **Input:** Question string `"{test_question}"`

2. **Dictionary mapping created:**
   - `retriever.invoke("{test_question}")` → retrieved 3 relevant chunks
   - `format_docs(chunks)` → formatted into single context string
   - `RunnablePassthrough()` → passed question through unchanged
   - **Result:** `{{"context": "...", "question": "{test_question}"}}`

3. **Prompt template filled in:**
   - Took the context and question
   - Filled in the `{{context}}` and `{{question}}` placeholders
   - **Result:** Complete prompt string ready for LLM

4. **LLM invoked:**
   - Sent prompt to ChatOpenAI (gpt-4o)
   - **Result:** AIMessage object with response

5. **Output parsed:**
   - `StrOutputParser()` extracted the text content
   - **Result:** Clean string answer (shown above)

**The power of RAG:** The answer is grounded in the retrieved pharmaceutical documents. The LLM didn't generate this from memory. Instead it read the relevant chunks and synthesized an answer.
"""

display(Markdown(output))


## RAG Chain Test

**Question:** "Is Zelomax safe during pregnancy?"

**Answer:**
Zelomax is not considered safe during pregnancy. It has been classified as very dangerous during a critical phase of pregnancy, with studies indicating decreased offspring viability and developmental anomalies. Its administration is contraindicated during certain periods of pregnancy.

---

**What just happened (step-by-step):**

1. **Input:** Question string `"Is Zelomax safe during pregnancy?"`

2. **Dictionary mapping created:**
   - `retriever.invoke("Is Zelomax safe during pregnancy?")` → retrieved 3 relevant chunks
   - `format_docs(chunks)` → formatted into single context string
   - `RunnablePassthrough()` → passed question through unchanged
   - **Result:** `{"context": "...", "question": "Is Zelomax safe during pregnancy?"}`

3. **Prompt template filled in:**
   - Took the context and question
   - Filled in the `{context}` and `{question}` placeholders
   - **Result:** Complete prompt string ready for LLM

4. **LLM invoked:**
   - Sent prompt to ChatOpenAI (gpt-4o)
   - **Result:** AIMessage object with response

5. **Output parsed:**
   - `StrOutputParser()` extracted the text content
   - **Result:** Clean string answer (shown above)

**The power of RAG:** The answer is grounded in the retrieved pharmaceutical documents. The LLM didn't generate this from memory. Instead it read the relevant chunks and synthesized an answer.


### Assemble Your RAG Chain

Now it's your turn! 

**Requirements:**
1. Create a variable named `rag_chain`. It should follow the exact template below:

```python
rag_chain = (
    {"context": ... | ..., "question": ...}
    | ...
    | ...
    | ...
)
```

2. Note that in the template above there are multiple placeholders (`...`). Replace each placeholder with the correct component, and in the right order. For reference, you can see review the cells above, as well as the primer. Your job is to chain togetherthe components using the pipe operator `|`.


**Hints:**
1. Start with the dictionary mapping for context and question
2. Pipe to the prompt template
3. Pipe to the LLM
4. Pipe to the output parser
5. Remember: Each `|` connects one component's output to the next component's input


In the code cell below, replace the line `raise NotImplementedError("Your code is missing.")` with your code. As this is an ungraded notebook, your work will not be submitted for grading. 

#### Enter Your Solution Then Run the Following Cell

In [12]:
# YOUR CODE HERE
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
# END OF YOUR CODE

# Test your chain
test_question = "What is Zelomax used for?"
try:
    answer = rag_chain.invoke(test_question)
    output = f"""
## Your RAG Chain Test

**Question:** "{test_question}"

**Answer:**
{answer}

---

**Complete chain breakdown:**

**What each pipe does:**

1. **`retriever | format_docs`** (inside dictionary)
   - Retrieves documents, pipes them to formatter
   - Returns formatted context string

2. **`dictionary | prompt`**
   - Takes `{{"context": "...", "question": "..."}}`
   - Pipes to PromptTemplate
   - Returns filled-in prompt

3. **`prompt | llm`**
   - Takes prompt string
   - Pipes to ChatOpenAI
   - Returns AIMessage object

4. **`llm | StrOutputParser()`**
   - Takes AIMessage
   - Pipes to parser
   - Returns clean string

**This is the core RAG pattern you'll use throughout your AI engineering development!**

"""
    display(Markdown(output))
except Exception as e:
    print(f"Error: {e}")
    print("Check your pipe operators - make sure each component is connected with |")


## Your RAG Chain Test

**Question:** "What is Zelomax used for?"

**Answer:**
I don't know.

---

**Complete chain breakdown:**

**What each pipe does:**

1. **`retriever | format_docs`** (inside dictionary)
   - Retrieves documents, pipes them to formatter
   - Returns formatted context string

2. **`dictionary | prompt`**
   - Takes `{"context": "...", "question": "..."}`
   - Pipes to PromptTemplate
   - Returns filled-in prompt

3. **`prompt | llm`**
   - Takes prompt string
   - Pipes to ChatOpenAI
   - Returns AIMessage object

4. **`llm | StrOutputParser()`**
   - Takes AIMessage
   - Pipes to parser
   - Returns clean string

**This is the core RAG pattern you'll use throughout your AI engineering development!**



<details>
<summary style="text-align: left; background-color: #e3fbe3; color: #2a8bc6; padding: 10px 10px; cursor: pointer;" role="alert"><strong>✅ Solution</strong></summary>

<p style="padding: 10px;">
<pre>
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser() 
)
</pre>
</p>

</details>

## Test Your RAG System

Now that you have a working RAG chain, try it with different questions!

This section demonstrates your RAG system in action. Just run the cells to see how well your system performs.

#### Run the Following Cell

In [13]:
# === Try different questions ===
questions = [
    "What are the contraindications for Zelomax?",
    "Who manufactures Xenthera?",
    "What is the recommended dosage of Yolonix?",
]

output = "## RAG System Test Results\n\n"

for i, q in enumerate(questions, 1):
    answer = rag_chain.invoke(q)
    output += f"""
### Question {i}: {q}

**Answer:** {answer}

---
"""

display(Markdown(output))

## RAG System Test Results


### Question 1: What are the contraindications for Zelomax?

**Answer:** Zelomax is contraindicated during the GD-06 to GD-19 period of pregnancy due to studies showing decreased offspring viability and skeletal anomalies. It is classified as very dangerous when administered during this critical phase of pregnancy.

---

### Question 2: Who manufactures Xenthera?

**Answer:** Xenthera is manufactured by Apex Pharma.

---

### Question 3: What is the recommended dosage of Yolonix?

**Answer:** I don't know.

---


## Reflection

Congratulations! You've built a complete RAG system using LangChain.

### What You Learned

**Key concepts:**
- **RAG:** Combining retrieval with generation
- **ChatOpenAI:** LangChain's wrapper for OpenAI's chat models
- **PromptTemplate:** Managing prompts with variables
- **StrOutputParser:** Extracting clean string outputs
- **Pipe operator (`|`):** Chaining components together
- **Runnable pattern:** How LangChain chains work

**The RAG chain pattern:**
```python
{"context": retriever | format_docs, "question": RunnablePassthrough()}
| prompt
| llm
| StrOutputParser()
```

This is the foundational pattern for building AI applications that need external knowledge.

### Questions to Consider

1. **Prompt engineering:** How might you modify the RAG prompt for different domains (legal, medical, technical)?

2. **Retrieval tuning:** What happens if you change `k=3` to `k=5` or `k=1`? How does this affect answer quality?

3. **Chunking strategy:** Our chunks are 500 characters. What are the trade-offs of smaller vs. larger chunks?

4. **Limitations:** What kinds of questions might this RAG system struggle with?

5. **Query ambiguity:** What if the user's question are too vague or too narrow? 

Keep these questions in mind as you learn about advanced RAG techniques.